In [ ]:
# Remove columns
import pandas as pd

csv_file = "all_59k_accidents_complete_final_2_cleaned.csv"  # Replace with your CSV path

# Read CSV
df = pd.read_csv(csv_file)

# Columns to remove
cols_to_drop = [
    "vehicletypebanded",
    "computed_day",
    "time_str"
]

# Drop only columns that exist
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Save back to the same file
df.to_csv(csv_file, index=False)

print(f"Updated CSV saved: {csv_file}")

In [ ]:
# Merge with weather for hourly
import logging
import pandas as pd

# =========================
# FILE PATHS
# =========================
file1_path = "all_59k_accidents_complete_final_2_cleaned.csv"
file2_path = "open_meteo_archive_missing_clean.csv"
output_path = "all_59k_accidents_complete_final_2_cleaned_updated.csv"

FILE1_LAT = "latitude"
FILE1_LON = "longitude"
FILE1_DT  = "dt"

FILE2_LAT = "source_latitude"
FILE2_LON = "source_longitude"
FILE2_DT  = "accident_datetime_utc"

# =========================
# JOIN KEYS
# =========================
KEY_LAT = "_k_lat"
KEY_LON = "_k_lon"
KEY_DT  = "_k_dt"
KEY_COLS = [KEY_LAT, KEY_LON, KEY_DT]

# =========================
# LOGGING
# =========================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("merge")

# =========================
# LOAD FILES
# =========================
logger.info("Loading files...")

df1 = pd.read_csv(file1_path, low_memory=False)
df2 = pd.read_csv(file2_path, low_memory=False)

logger.info(f"file1 rows={len(df1):,}")
logger.info(f"file2 rows={len(df2):,}")

# =========================
# NORMALIZATION
# =========================
def norm_latlon(s):
    return (
        pd.to_numeric(s, errors="coerce")
        .map(lambda x: format(x, ".10f") if pd.notnull(x) else None)
    )

def norm_dt(s):
    return (
        pd.to_datetime(s, errors="coerce", utc=True)
        .dt.strftime("%Y-%m-%dT%H:%M:%S+00:00")
    )

df1[KEY_LAT] = norm_latlon(df1[FILE1_LAT])
df1[KEY_LON] = norm_latlon(df1[FILE1_LON])
df1[KEY_DT]  = norm_dt(df1[FILE1_DT])

df2[KEY_LAT] = norm_latlon(df2[FILE2_LAT])
df2[KEY_LON] = norm_latlon(df2[FILE2_LON])
df2[KEY_DT]  = norm_dt(df2[FILE2_DT])

# =========================
# SHARED COLUMNS
# =========================
shared_cols = sorted(
    (set(df1.columns) & set(df2.columns))
    - set(KEY_COLS)
)

logger.info(f"Shared columns found: {len(shared_cols)}")

# =========================
# DEDUPE FILE2 BY KEY
# =========================
# Keeps one record per normalized key.
# Then every matching row in file1 gets those values.
df2_lookup = (
    df2[KEY_COLS + shared_cols]
    .drop_duplicates(subset=KEY_COLS)
)

logger.info(
    f"file2 unique key rows={len(df2_lookup):,}"
)

# =========================
# MERGE
# =========================
logger.info("Merging...")

merged = df1.merge(
    df2_lookup,
    on=KEY_COLS,
    how="left",
    indicator=True,
    validate="many_to_one",
    suffixes=("", "_src")
)

matched = merged["_merge"].eq("both")

matched_rows = int(matched.sum())
unmatched_rows = int((~matched).sum())

logger.info(f"Matched rows: {matched_rows:,}")
logger.info(f"Unmatched rows: {unmatched_rows:,}")

# =========================
# COPY VALUES
# =========================
total_replacements = 0
columns_changed = []

for col in shared_cols:

    src_col = f"{col}_src"

    # Only replace when file2 actually has a value
    replace_mask = matched & merged[src_col].notna()

    before = merged.loc[replace_mask, col]
    after  = merged.loc[replace_mask, src_col]

    changed_mask = ~(
        before.eq(after)
        | (before.isna() & after.isna())
    )

    replacements = int(changed_mask.sum())

    merged.loc[replace_mask, col] = after

    total_replacements += replacements
    columns_changed.append((col, replacements))

    logger.info(
        f"Column '{col}': {replacements:,} replacements"
    )

logger.info(
    f"Total replacements made: {total_replacements:,}"
)

# =========================
# CLEANUP
# =========================
helper_cols = (
    KEY_COLS
    + ["_merge"]
    + [f"{c}_src" for c in shared_cols]
)

updated_df = merged.drop(
    columns=[c for c in helper_cols if c in merged.columns]
)

# =========================
# SAVE
# =========================
updated_df.to_csv(output_path, index=False)

logger.info(f"Saved: {output_path}")

# =========================
# SUMMARY
# =========================
print("\n==============================")
print("SYNC SUMMARY")
print("==============================")

print(f"Rows in file1:      {len(df1):,}")
print(f"Rows in file2:      {len(df2):,}")
print(f"Matched rows:       {matched_rows:,}")
print(f"Unmatched rows:     {unmatched_rows:,}")
print(f"Total replacements: {total_replacements:,}")

print("\nPER-COLUMN REPLACEMENTS")
print("-----------------------")

for col, n in sorted(
    columns_changed,
    key=lambda x: x[1],
    reverse=True
):
    print(f"{col}: {n:,}")

In [ ]:
# Convert jsonl to csv
#!/usr/bin/env python3

import json
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

INPUT_JSONL = "open_meteo_archive_missing.jsonl"
OUTPUT_CSV = "open_meteo_archive_missing.csv"


# ============================================================
# HELPERS
# ============================================================

def first_or_none(value):
    if isinstance(value, list):
        return value[0] if value else None
    return value


# ============================================================
# MAIN
# ============================================================

rows = []

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        obj = json.loads(line)

        row = {}

        # ====================================================
        # TOP-LEVEL FIELDS
        # ====================================================

        for k, v in obj.items():
            if k == "archive":
                continue

            row[k] = v

        # ====================================================
        # ARCHIVE SECTION
        # ====================================================

        archive = obj.get("archive", {})

        for k, v in archive.items():

            # Skip nested hourly for separate handling
            if k == "hourly":
                continue

            # Skip hourly_units metadata
            if k == "hourly_units":
                continue

            row[f"archive_{k}"] = v

        # ====================================================
        # HOURLY WEATHER VARIABLES
        # ====================================================

        hourly = archive.get("hourly", {})

        for k, v in hourly.items():
            row[k] = first_or_none(v)

        rows.append(row)

# ============================================================
# EXPORT
# ============================================================

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(df):,} rows")
print(f"Saved CSV -> {OUTPUT_CSV}")
print(f"Columns -> {len(df.columns)}")

print("\nColumns:")
for c in df.columns:
    print("-", c)

In [ ]:
# Remove obj detection cols with all 0 values
import pandas as pd
from google.colab import files
from IPython.display import display

# Upload CSV
uploaded = files.upload()
file_name = next(iter(uploaded))

# Read CSV
df = pd.read_csv(file_name)

# Find columns whose names start with 'obj_'
obj_cols = [c for c in df.columns if str(c).startswith("obj_")]

# Build summary table
summary = []
for col in obj_cols:
    numeric_col = pd.to_numeric(df[col], errors="coerce").fillna(0)
    all_zero = (numeric_col == 0).all()

    summary.append({
        "column_name": col,
        "status": "removed" if all_zero else "kept",
        "reason": "all values are 0" if all_zero else "has non-zero values"
    })

summary_df = pd.DataFrame(summary)

# Show the table
display(summary_df)

# Save for easy copy into Google Docs / Sheets
summary_df.to_csv("obj_columns_summary.csv", index=False)

# Also print as text table in case you want to copy/paste
print(summary_df.to_string(index=False))

files.download("obj_columns_summary.csv")